# 01 — Fine-tune Micro-SAM for TEM Layer Detection

**Goal:** Fine-tune Micro-SAM ViT-B LM on TEM cross-section images so that a single click on the layer predicts the layer mask accurately.

**Environment:** Google Colab (GPU). Runtime → Change runtime type → T4 GPU.

**Data needed:** Upload `training_data/layer_detection/` to Google Drive (or directly to Colab).

## 1. Install dependencies

In [ ]:
!pip install micro-sam>=1.0
!pip install tifffile imagecodecs

## 2. Load training data

Upload your `training_data/layer_detection/` folder to Colab (or mount Google Drive).

Expected structure:
```
layer_detection/
├── images/       # full TEM cross-section TIFFs
├── masks/        # binary layer masks (0/1)
├── points.json   # click points {"sample_id": [col, row]}
└── train_val_split.json
```

In [ ]:
import os, json, numpy as np, tifffile
from pathlib import Path

# Set this to your data path.
DATA_DIR = Path('/content/layer_detection')  # or '/content/drive/MyDrive/layer_detection'

# Load split.
with open(DATA_DIR / 'train_val_split.json') as f:
    split = json.load(f)

# Load click points.
with open(DATA_DIR / 'points.json') as f:
    points = json.load(f)

print(f'Train: {len(split["train"])}, Val: {len(split["val"])}')
print(f'Click points: {len(points)}')

# Preview one sample.
sample_id = split['train'][0]
img = tifffile.imread(DATA_DIR / 'images' / f'{sample_id}.tif')
mask = tifffile.imread(DATA_DIR / 'masks' / f'{sample_id}.tif')
print(f'Image: {img.shape} {img.dtype}, Mask: {mask.shape} {mask.dtype}')
print(f'Click point: {points.get(sample_id, "N/A")}')

## 3. Prepare micro-sam training format

Micro-SAM expects data in a specific format for `micro_sam.finetune`. We create a dataset of (image, mask, point_prompt) triplets.

In [ ]:
import torch
from micro_sam.util import get_sam_model

# Download the pre-trained Micro-SAM ViT-B LM model (~350 MB on first run).
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

predictor = get_sam_model(model_type='vit_b_lm', device=device)
print('Model loaded.')

## 4. Fine-tune

Fine-tuning keeps the image encoder frozen and updates the prompt encoder +
mask decoder on your TEM layer data — fast (roughly 10–20 min for 20 epochs
on a T4 GPU). Full-encoder tuning or PEFT/LoRA is available via the
`freeze` / `peft_kwargs` arguments of `train_sam`.

In [ ]:
# Build lists of (image, mask) pairs for training.
# micro-sam's training API takes raw images + label images; the point/box
# prompts used during interactive training are generated automatically
# from the labels (points.json is not needed for fine-tuning).
train_images, train_masks = [], []
for sample_id in split['train']:
    img = tifffile.imread(DATA_DIR / 'images' / f'{sample_id}.tif')
    mask = tifffile.imread(DATA_DIR / 'masks' / f'{sample_id}.tif')
    if img is None or mask is None:
        continue
    train_images.append(img)
    train_masks.append(mask.astype(np.int32))

val_images, val_masks = [], []
for sample_id in split['val']:
    img = tifffile.imread(DATA_DIR / 'images' / f'{sample_id}.tif')
    mask = tifffile.imread(DATA_DIR / 'masks' / f'{sample_id}.tif')
    if img is None or mask is None:
        continue
    val_images.append(img)
    val_masks.append(mask.astype(np.int32))

print(f'Train samples: {len(train_images)}, Val samples: {len(val_images)}')

In [ ]:
# Fine-tune with the current micro-sam training API:
# https://github.com/computational-cell-analytics/micro-sam
import os
from torch.utils.data import DataLoader
from micro_sam.training import train_sam, default_sam_dataset

CHECKPOINT_PATH = '/content/microsam_tem_layer.pth'  # pre-trained init weights

# micro-sam trains on patches sampled from the images (batch_size=1 is the
# micro-sam standard). The extra instance-segmentation decoder is not needed
# for our point-prompt use case, so we train without it.
train_ds = default_sam_dataset(
    raw_paths=train_images, raw_key=None,
    label_paths=train_masks, label_key=None,
    patch_shape=(512, 512), with_segmentation_decoder=False, is_train=True,
)
val_ds = default_sam_dataset(
    raw_paths=val_images, raw_key=None,
    label_paths=val_masks, label_key=None,
    patch_shape=(512, 512), with_segmentation_decoder=False, is_train=False,
)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

train_sam(
    name='microsam_tem_layer',
    model_type='vit_b_lm',
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=20,
    lr=1e-5,
    checkpoint_path=CHECKPOINT_PATH,   # initialize from pre-trained vit_b_lm
    freeze=['image_encoder'],          # encoder frozen — fast on a T4
    device=device,
)

# micro-sam saves checkpoints to 'checkpoints/<name>' under save_root (cwd).
BEST = os.path.join('checkpoints', 'microsam_tem_layer', 'best.pth')
print(f'Fine-tuned checkpoint: {os.path.abspath(BEST)}')

## 5. Evaluate on validation set

In [ ]:
from micro_sam.util import get_sam_model

# Load the fine-tuned model.
predictor_ft = get_sam_model(model_type='vit_b_lm', checkpoint_path=BEST,
                             device=device)

# Evaluate on each validation sample: prompt with the centroid of the GT
# mask (always inside the layer), measure IoU vs. the full GT mask.
ious = []
for i in range(len(val_images)):
    img = val_images[i]
    gt_mask = val_masks[i] > 0
    rows, cols = np.nonzero(gt_mask)
    point = [int(cols.mean()), int(rows.mean())]  # [x, y] = [col, row]

    # Predict with point prompt.
    predictor_ft.set_image(img)
    masks, scores, _ = predictor_ft.predict(
        point_coords=np.array([point]),
        point_labels=np.array([1]),  # 1 = foreground
        multimask_output=False,
    )
    pred_mask = masks[0] > 0

    # IoU.
    intersection = (pred_mask & gt_mask).sum()
    union = (pred_mask | gt_mask).sum()
    iou = intersection / max(union, 1)
    ious.append(iou)
    print(f'Val #{i+1}: IoU = {iou:.3f}')

print(f'\nMean IoU: {np.mean(ious):.3f} ± {np.std(ious):.3f}')

## 6. Download the fine-tuned checkpoint

Download `checkpoints/microsam_tem_layer/best.pth` and place the file in
`~/.cache/tem_grain_analyzer/` on your machine. To expose it in the Streamlit
app's **SAM model** dropdown, add a registry entry for it in
`apps/02_grain_analyzer/src/sam_segmenter.py` (see `MODEL_REGISTRY`);
a custom checkpoint can always be loaded directly via
`get_sam_model(model_type='vit_b_lm', checkpoint_path=...)`.

In [ ]:
from google.colab import files
files.download(os.path.abspath(BEST))